# Split par centre
# Sélection manuelle des centres test (~20% du dataset). Le split est ensuite exporté en `data/train_df.csv` et `data/test_df.csv` sans la colonne `Center`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path("../..").resolve()
sys.path.append(str(ROOT / "src"))

DATA_OLD   = ROOT / "data" / "dataframe_cleaned_old.csv"
DATA_NEW   = ROOT / "data" / "dataframe_cleaned_2_with_center.csv"
OUT_TRAIN  = ROOT / "data" / "train_df.csv"
OUT_TEST   = ROOT / "data" / "test_df.csv"
TARGET_COL = "outcome"

df = pd.read_csv(DATA_OLD, sep=";")
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

centers = pd.read_csv(DATA_NEW, sep=";")[["Order", "Center"]]
df = df.merge(centers, on="Order", how="left")

print(f"Dataset: {df.shape[0]} patients, {df['Center'].isna().sum()} sans centre")


Dataset: 4841 patients, 0 sans centre


## Distribution des centres

In [2]:
center_stats = (
    df.groupby("Center")
    .agg(
        n=("Center", "count"),
        pct=("Center", lambda x: round(len(x) / len(df) * 100, 1)),
        outcome_rate=(TARGET_COL, lambda x: round(x.mean(), 3)),
    )
    .sort_values("n", ascending=False)
)
print(f"Total: {len(df)} patients\n")
center_stats


Total: 4841 patients



,n,pct,outcome_rate
Center,,,
DE (Berlin),1097,22.7,0.493
BE (Bordet),652,13.5,0.383
IT (Turin),463,9.6,0.354
IT (Regina),448,9.3,0.520
Brest,369,7.6,0.680
FR (Reims),340,7.0,0.429
CZ (Ostrava),267,5.5,0.326
FR (Bordeaux),261,5.4,0.510
FR (Grenoble),233,4.8,0.451


## Sélection des centres test\nModifie `TEST_CENTERS` avec les noms exacts de la colonne `Center` ci-dessus.

In [3]:
TEST_CENTERS = [
     "FR (Bordeaux)",
     "FR (Toulouse)",
     "CZ (Ostrava)",
     "FR (Grenoble)",
]

selected = center_stats.loc[center_stats.index.isin(TEST_CENTERS)]
print(f"Centres sélectionnés : {len(TEST_CENTERS)}")
print(f"Test set : {selected['n'].sum()} patients ({selected['pct'].sum():.1f}%)\n")
selected


Centres sélectionnés : 4
Test set : 962 patients (19.9%)



,n,pct,outcome_rate
Center,,,
CZ (Ostrava),267,5.5,0.326
FR (Bordeaux),261,5.4,0.510
FR (Grenoble),233,4.8,0.451
FR (Toulouse),201,4.2,0.552


## Split & vérification

In [4]:
assert TEST_CENTERS, "TEST_CENTERS est vide — choisis au moins un centre."
unknown = [c for c in TEST_CENTERS if c not in df["Center"].values]
assert not unknown, f"Centres introuvables dans le dataset : {unknown}"

test_mask = df["Center"].isin(TEST_CENTERS)
test_df  = df[test_mask].reset_index(drop=True)
train_df = df[~test_mask].reset_index(drop=True)

print("=== Tailles ===")
print(f"Train : {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Test  : {len(test_df)}  ({len(test_df)/len(df)*100:.1f}%)")

print("\n=== Distribution target ===")
print(f"Train outcome rate : {train_df[TARGET_COL].mean():.3f}")
print(f"Test  outcome rate : {test_df[TARGET_COL].mean():.3f}")


=== Tailles ===
Train : 3879 (80.1%)
Test  : 962  (19.9%)

=== Distribution target ===
Train outcome rate : 0.460
Test  outcome rate : 0.453


## Optionnel : drop centres & features

In [5]:
# Centres à exclure complètement du dataset (ni train ni test)
CENTERS_TO_DROP = [
    # "CH (Geneve)",  # Outliers sur trop de features
]

if CENTERS_TO_DROP:
    before = len(train_df) + len(test_df)
    train_df = train_df[~train_df["Center"].isin(CENTERS_TO_DROP)].reset_index(drop=True)
    test_df  = test_df[~test_df["Center"].isin(CENTERS_TO_DROP)].reset_index(drop=True)
    print(f"Dropped centres: {CENTERS_TO_DROP} → {before - len(train_df) - len(test_df)} rows removed")
else:
    print("Aucun centre droppé.")


Aucun centre droppé.


In [ ]:
train_out = train_df.drop(columns=["Center"], errors="ignore")
test_out  = test_df.drop(columns=["Center"], errors="ignore")

## Export

In [ ]:
train_out.to_csv(OUT_TRAIN, index=False, sep=";")
test_out.to_csv(OUT_TEST,   index=False, sep=";")

print(f"Saved: {OUT_TRAIN}  {train_out.shape}")
print(f"Saved: {OUT_TEST}   {test_out.shape}")
print(f"\nTest centres : {TEST_CENTERS} avec {selected['n'].sum()} patients ({selected['pct'].sum():.1f}%)")
print(f"Dropped centres : {CENTERS_TO_DROP}")